# Stage 9D — Make the new chronological split

**Goal:** Combine the 485 prepared daily files into training, validation, and sealed test files.

**Why:** We use 2025 to train, January–February 2026 to choose the next model, and March–April 2026 for one final evaluation. This test period was not used in the first model's diagnosis.

The code verifies every prepared file, checks IDs and fields, writes three split files, and records their checksums. It prints label balance only for training and validation; test outcomes remain sealed for model selection.

In [ ]:
# Cell 2 — Why: pin the prepared sample and coverage plan.
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
prepared_id = "20260926T142035821748Z"
prepared_file = root / "prepared_datasets" / prepared_id / "manifest.json"
prepared = json.loads(prepared_file.read_text())
plan_id = "20260926T130607938949Z"
plan_file = root / "release_decisions" / plan_id / "coverage_plan.json"
plan = json.loads(plan_file.read_text())
if prepared["prepared_id"] != prepared_id or len(prepared["parts"]) != 485:
    raise ValueError("Wrong prepared dataset")
if prepared["totals"]["saved"] != sum(part["rows"] for part in prepared["parts"]):
    raise ValueError("Prepared row count does not match daily parts")
print("Prepared dataset:", prepared_id, "Labeled rows:", prepared["totals"]["saved"])

In [ ]:
# Cell 3 — Why: assign a day by time, never by its outcome.
windows = plan["windows_start_inclusive_end_exclusive"]

def group_for(day):
    if windows["train"][0] <= day < windows["train"][1]:
        return "train"
    if windows["validation"][0] <= day < windows["validation"][1]:
        return "validation"
    if windows["test"][0] <= day < windows["test"][1]:
        return "test"
    raise ValueError(f"Day outside the approved plan: {day}")

days = [part["day"] for part in prepared["parts"]]
if days != sorted(set(days)) or len(days) != plan["daily_partitions_planned"]:
    raise ValueError("Daily dates are missing, repeated, or out of order")
print("Window counts:", {name: sum(group_for(day) == name for day in days)
                          for name in ["train", "validation", "test"]})

In [ ]:
# Cell 4 — Why: create a fresh split ID and temporary output files.
from contextlib import ExitStack

split_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
split_dir = root / "splits" / split_id
split_dir.mkdir(parents=True, exist_ok=False)
names = ["train", "validation", "test"]
counts = {name: 0 for name in names}
visible_labels = {name: {0: 0, 1: 0} for name in ["train", "validation"]}
seen_ids = set()
expected_fields = {"unique_key", "created_date", "complaint_type", "borough",
                   "month", "hour", "weekday", "label"}
print("New split ID:", split_id)

In [ ]:
# Cell 5 — Why: verify every day and write each row to its time group.
with ExitStack() as stack:
    outputs = {name: stack.enter_context(open(split_dir / f"{name}.jsonl.pending", "wb"))
               for name in names}
    for number, part in enumerate(prepared["parts"], 1):
        raw = (root / part["file"]).read_bytes()
        if hashlib.sha256(raw).hexdigest() != part["sha256"] or len(raw.splitlines()) != part["rows"]:
            raise ValueError(f"{part['day']}: prepared file changed")
        group = group_for(part["day"])
        for line in raw.splitlines():
            row = json.loads(line)
            if set(row) != expected_fields or row["label"] not in [0, 1]:
                raise ValueError(f"{part['day']}: invalid fields or label")
            if str(row["created_date"])[:10] != part["day"]:
                raise ValueError(f"{part['day']}: wrong creation day")
            request_id = str(row["unique_key"])
            if request_id in seen_ids:
                raise ValueError(f"Duplicate request ID across days: {request_id}")
            seen_ids.add(request_id)
            outputs[group].write(line + b"\n")
            counts[group] += 1
            if group != "test":
                visible_labels[group][row["label"]] += 1
        if number % 50 == 0 or number == len(days):
            print("Verified and assigned days:", number, "of", len(days), flush=True)
print("Rows by split:", counts)
print("Training/validation label counts:", visible_labels)

In [ ]:
# Cell 6 — Why: verify saved files before publishing the split manifest.
if sum(counts.values()) != prepared["totals"]["saved"] or len(seen_ids) != sum(counts.values()):
    raise ValueError("Split lost or duplicated rows")
files = {}
for name in names:
    pending = split_dir / f"{name}.jsonl.pending"
    content = pending.read_bytes()
    if len(content.splitlines()) != counts[name]:
        raise ValueError(f"{name}: saved row count differs")
    final_file = split_dir / f"{name}.jsonl"
    pending.replace(final_file)
    files[name] = {"rows": counts[name], "sha256": hashlib.sha256(final_file.read_bytes()).hexdigest()}
print("Verified split files:", {name: info["rows"] for name, info in files.items()})

In [ ]:
# Cell 7 — Why: save lineage and keep the test period sealed.
split_manifest = {
    "split_id": split_id, "source_prepared_id": prepared_id,
    "source_prepared_manifest_sha256": hashlib.sha256(prepared_file.read_bytes()).hexdigest(),
    "source_dataset_id": prepared["source_dataset_id"],
    "coverage_plan_id": plan_id,
    "coverage_plan_sha256": hashlib.sha256(plan_file.read_bytes()).hexdigest(),
    "windows_start_inclusive_end_exclusive": windows,
    "sample_rule": prepared["sample_rule"],
    "test_sealed_for_model_selection": True,
    "features": ["complaint_type", "borough", "month", "hour", "weekday"],
    "label_rule": "1 if not closed within 72 actual hours; else 0",
    "files": files, "training_validation_labels": visible_labels,
    "lesson_version": "stage9d-v1"
}
(split_dir / "manifest.json").write_text(json.dumps(split_manifest, indent=2) + "\n", encoding="utf-8")
print("Split ID:", split_id)
print("Saved under:", split_dir)
print("Test rows saved; test label rate not displayed")

# Stage 9D takeaway and quiz

**What you learned:** The split is chronological, complete, and traceable to the full raw dataset through the prepared dataset ID. We checked every row's date and request ID and created a checksum for each split file. The model will use validation to select settings and touch test labels only after that choice.

**Easy quiz:**

1. Why assign by creation date instead of random rows?
2. Why check request IDs across all 485 days?
3. Why can we print train and validation label counts but keep test label counts sealed?
4. What does `source_prepared_id` let us trace?

Send me **Window counts**, **Rows by split**, **Training/validation label counts**, and **Split ID**. Next we will train and compare models on this broader time coverage.